#### Import Libraries

In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

#### Loading the Dataset

In [3]:
# Loading the dataset
df = pd.read_csv("../data/raw/defects_data.csv")

#### Displaying the first 10 rows of the dataset to get a quick look at the data

In [4]:
df.head(10)

,defect_id,product_id,defect_type,defect_date,defect_location,severity,inspection_method,repair_cost
0,1,15,Structural,6/6/2024,Component,Minor,Visual Inspection,245.47
1,2,6,Functional,4/26/2024,Component,Minor,Visual Inspection,26.87
2,3,84,Structural,2/15/2024,Internal,Minor,Automated Testing,835.81
3,4,10,Functional,3/28/2024,Internal,Critical,Automated Testing,444.47
4,5,14,Cosmetic,4/26/2024,Component,Minor,Manual Testing,823.64
5,6,17,Functional,5/11/2024,Internal,Moderate,Visual Inspection,788.11
6,7,85,Cosmetic,5/23/2024,Internal,Critical,Manual Testing,33.68
7,8,90,Structural,1/15/2024,Internal,Moderate,Manual Testing,65.56
8,9,30,Structural,1/26/2024,Component,Critical,Manual Testing,848.61
9,10,20,Structural,6/5/2024,Component,Critical,Visual Inspection,478.48


#### Displaying the total number of rows and columns in the dataset

In [5]:
rows, columns = df.shape
print(f'The dataset contains {rows} rows and {columns} columns.')

The dataset contains 1000 rows and 8 columns.


#### Displaying column names

In [6]:
df.columns

Index(['defect_id', 'product_id', 'defect_type', 'defect_date',
       'defect_location', 'severity', 'inspection_method', 'repair_cost'],
      dtype='str')

#### Based on the dataset documentation, the meaning of each column in the Manufacturing Defects dataset is as follows:

- **defect_id** - Unique identifier assigned to each defect record.
- **product_id** - Identifier of the product associated with the defect.
- **defect_type** - Type or category of defect, such as cosmetic, functional, or structural.
- **defect_date** - Date on which the defect was detected.
- **defect_location** - Location within the product where the defect was identified, such as the surface or component.
- **severity** - Severity level of the defect, categorized as Minor, Moderate, or Critical, and serving as the primary target variable for multiclass classification.
- **inspection_method** - Method used to detect or identify the defect, such as visual inspection, automated testing, or manual testing.
- **repair_cost** - Cost incurred to repair or address the identified defect.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   defect_id          1000 non-null   int64  
 1   product_id         1000 non-null   int64  
 2   defect_type        1000 non-null   str    
 3   defect_date        1000 non-null   str    
 4   defect_location    1000 non-null   str    
 5   severity           1000 non-null   str    
 6   inspection_method  1000 non-null   str    
 7   repair_cost        1000 non-null   float64
dtypes: float64(1), int64(2), str(5)
memory usage: 110.4 KB


In [8]:
#### Displaying descriptive statistics for numerical variables

df.describe().T

,count,mean,std,min,25%,50%,75%,max
defect_id,1000.0,500.50000,288.819436,1.00,250.7500,500.50,750.250,1000.00
product_id,1000.0,50.83700,29.480935,1.00,26.0000,51.00,77.000,100.00
repair_cost,1000.0,507.62715,289.623615,10.22,270.9025,506.43,759.065,999.64


In [9]:
#### Displaying columns along with their data types

print(df.dtypes)

defect_id              int64
product_id             int64
defect_type              str
defect_date              str
defect_location          str
severity                 str
inspection_method        str
repair_cost          float64
dtype: object


#### Observations on Data Types

- The dataset contains a combination of numerical and categorical variables representing defect identifiers, product information, defect characteristics, inspection methods, and repair costs.
- Numerical variables are stored using int64 and float64 data types, while defect_type, defect_date, defect_location, severity, and inspection_method are stored as string (str) variables.
- The current data types are generally appropriate for initial analysis, although defect_date is stored as a string and may require date conversion during data preparation for machine learning.

In [10]:
#### Checking unique values for each column

for col in df.columns:
    unique_vals = df[col].unique()

    print(f'Column Name : {col}')
    print(f'Number of Unique Values : {len(unique_vals)}')
    print(f'Unique Values : {unique_vals}')
    print('-' * 50)

Column Name : defect_id
Number of Unique Values : 1000
Unique Values : [   1    2    3    4    5    6    7    8    9   10   11   12   13   14
   15   16   17   18   19   20   21   22   23   24   25   26   27   28
   29   30   31   32   33   34   35   36   37   38   39   40   41   42
   43   44   45   46   47   48   49   50   51   52   53   54   55   56
   57   58   59   60   61   62   63   64   65   66   67   68   69   70
   71   72   73   74   75   76   77   78   79   80   81   82   83   84
   85   86   87   88   89   90   91   92   93   94   95   96   97   98
   99  100  101  102  103  104  105  106  107  108  109  110  111  112
  113  114  115  116  117  118  119  120  121  122  123  124  125  126
  127  128  129  130  131  132  133  134  135  136  137  138  139  140
  141  142  143  144  145  146  147  148  149  150  151  152  153  154
  155  156  157  158  159  160  161  162  163  164  165  166  167  168
  169  170  171  172  173  174  175  176  177  178  179  180  181  182
  183 

In [11]:
#### Checking the target variable distribution

print("Severity Value Counts:")
print(df['severity'].value_counts())

print("\nSeverity Percentage:")
print(df['severity'].value_counts(normalize=True) * 100)

Severity Value Counts:
severity
Minor       358
Critical    333
Moderate    309
Name: count, dtype: int64

Severity Percentage:
severity
Minor       35.8
Critical    33.3
Moderate    30.9
Name: proportion, dtype: float64


In [12]:
#### Checking missing values per column, including empty strings

# Count of NaN / None values per column
nan_count = df.isnull().sum()

# Count of empty strings per column
empty_count = (df == '').sum()

# Total missing values
total_missing = nan_count + empty_count

# Percentage of missing values
missing_percentage = (total_missing / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing Values': total_missing,
    'Percentage (%)': missing_percentage
}, index=df.columns).sort_values(
    by='Percentage (%)',
    ascending=False
)

print(missing_df)

                   Missing Values  Percentage (%)
defect_id                       0             0.0
product_id                      0             0.0
defect_type                     0             0.0
defect_date                     0             0.0
defect_location                 0             0.0
severity                        0             0.0
inspection_method               0             0.0
repair_cost                     0             0.0


In [13]:
#### Checking for duplicated rows

df.duplicated().sum()

np.int64(0)

#### Numerical Data Validation


In [14]:
#### Numerical Data Validation

numerical_columns = ['defect_id', 'product_id', 'repair_cost']

df[numerical_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
defect_id,1000.0,500.50000,288.819436,1.00,250.7500,500.50,750.250,1000.00
product_id,1000.0,50.83700,29.480935,1.00,26.0000,51.00,77.000,100.00
repair_cost,1000.0,507.62715,289.623615,10.22,270.9025,506.43,759.065,999.64


#### Checking for negative values

In [15]:
negative_values = (df[numerical_columns] < 0).sum()

print("Negative Values:")
print(negative_values)

Negative Values:
defect_id      0
product_id     0
repair_cost    0
dtype: int64


#### Checking categorical variables

In [16]:
#### Checking categorical variables

categorical_columns = [
    'defect_type',
    'defect_location',
    'severity',
    'inspection_method'
]

for col in categorical_columns:
    print(f'Column Name : {col}')
    print(f'Number of Unique Values : {df[col].nunique()}')
    print(f'Unique Values : {df[col].unique()}')
    print('-' * 50)

Column Name : defect_type
Number of Unique Values : 3
Unique Values : <ArrowStringArray>
['Structural', 'Functional', 'Cosmetic']
Length: 3, dtype: str
--------------------------------------------------
Column Name : defect_location
Number of Unique Values : 3
Unique Values : <ArrowStringArray>
['Component', 'Internal', 'Surface']
Length: 3, dtype: str
--------------------------------------------------
Column Name : severity
Number of Unique Values : 3
Unique Values : <ArrowStringArray>
['Minor', 'Critical', 'Moderate']
Length: 3, dtype: str
--------------------------------------------------
Column Name : inspection_method
Number of Unique Values : 3
Unique Values : <ArrowStringArray>
['Visual Inspection', 'Automated Testing', 'Manual Testing']
Length: 3, dtype: str
--------------------------------------------------


#### Validating the target variable

In [17]:
#### Validating the target variable

print("Target Variable : severity")
print(f"Number of Classes : {df['severity'].nunique()}")

print("\nTarget Classes:")
print(df['severity'].unique())

print("\nClass Distribution:")
print(df['severity'].value_counts())

print("\nClass Distribution (%):")
print((df['severity'].value_counts(normalize=True) * 100).round(2))

Target Variable : severity
Number of Classes : 3

Target Classes:
<ArrowStringArray>
['Minor', 'Critical', 'Moderate']
Length: 3, dtype: str

Class Distribution:
severity
Minor       358
Critical    333
Moderate    309
Name: count, dtype: int64

Class Distribution (%):
severity
Minor       35.8
Critical    33.3
Moderate    30.9
Name: proportion, dtype: float64


In [18]:
#### Initial Data Quality Checks

print("Dataset Shape:", df.shape)
print("Total Missing Values:", df.isnull().sum().sum())
print("Total Duplicate Rows:", df.duplicated().sum())
print("Total Features:", df.shape[1] - 1)
print("Target Variable:", 'severity')
print("Number of Target Classes:", df['severity'].nunique())

Dataset Shape: (1000, 8)
Total Missing Values: 0
Total Duplicate Rows: 0
Total Features: 7
Target Variable: severity
Number of Target Classes: 3


#### Observations on Initial Data Quality

- The dataset contains **1,000 records and 8 columns**, including 7 potential predictor variables and 1 target variable (`severity`).
- No missing values or duplicate records were identified, indicating that the dataset is complete at the initial validation stage.
- The numerical variables contain no negative values, and the categorical variables contain consistent and expected categories.
- The target variable `severity` contains three classes - Minor, Moderate, and Critical — with a relatively balanced distribution across the dataset.
- `defect_id` functions as a unique record identifier, while `product_id` represents products associated with multiple defect records.
- `defect_date` is currently stored as a string and will require appropriate date handling during data preparation.